# Services, DNS & Service Discovery

## What's covered

- **Why Services exist** — pods come and go, their IPs change, and clients can't keep up
- **The Service abstraction** — stable name plus virtual IP, backed by a label selector that resolves to whichever pods match right now
- **`port`, `targetPort`, `nodePort`** — the three numbers in a Service spec and which is which
- **Endpoints and EndpointSlices** — the actual list of pod IPs behind a Service, maintained by a controller
- **Service types** — `ClusterIP`, `NodePort`, `LoadBalancer`, `ExternalName`, and when each one fits
- **`kube-proxy`** — how Service IPs actually route to pods on every node (iptables, IPVS, nftables)
- **Cluster DNS** — CoreDNS, the `<svc>.<ns>.svc.cluster.local` name pattern, and how pods resolve it
- **Headless Services** — no virtual IP, DNS returns pod IPs directly; the building block under StatefulSets
- **Session affinity** — `ClientIP` for sticky routing
- **Multi-port services** — naming ports so manifests stay readable
- **Service vs Ingress** — a glance at what comes in notebook 09

By the end of this notebook you should be able to expose a Deployment, reach it from another pod by DNS, pick the right Service type for each problem, and explain what `kube-proxy` and CoreDNS are doing under the hood. Notebook 03 is the prerequisite — you should have a Deployment manifest in your head, because every Service in real life sits in front of one.

## Why Services exist

You finished notebook 03 with a Deployment running three nginx pods. Each pod got its own IP. Pods come, pods go — every restart, every rollout, every scale event mints new IPs.

Now you want a *web* pod to call your *api*. The web pod can't:

- **Hardcode pod IPs** — they change every rollout.
- **Discover api pods by listing all pods** — it would have to watch the API server, filter by label, retry on changes. That's a tiny custom controller per client.
- **Use DNS to resolve `api-7c9d-x2k5p`** — there's no DNS for individual pods by default, and even if there were, that name has a random suffix.

What every client really wants is a **stable name** and a **stable address** that always point at "whichever api pods are alive right now." That's the **Service**.

A Service is a tiny piece of glue between *consumers* and *producers*:

```
       web pod                        3x api pods
   +-------------+              +-------+ +-------+ +-------+
   |             | GET http://  | api-1 | | api-2 | | api-3 |
   |   client    |   api:80     +-------+ +-------+ +-------+
   |             |------------> pod IPs change every rollout
   +-------------+              +-------+ +-------+ +-------+
        |                       ^
        |  DNS: api -> 10.96.0.42 (a stable virtual IP)
        |                       |
        +---------> kube-proxy + iptables/IPVS forward
                    to a healthy pod IP
```

The web pod calls `http://api:80`. DNS resolves `api` to a stable virtual IP. `kube-proxy` rewrites the packet to one of the pod IPs. The web pod has no idea — and no need to know — which specific pod served the request, or that the api Deployment just finished a rollout.

## The Service abstraction — stable identity by label selector

A **Service** is a Kubernetes object with two halves:

- A **virtual IP and name** — clients connect here, this never changes.
- A **selector** — `app: api`, the same label query you met in notebook 02.

A controller — the **endpoints controller**, technically — runs the loop:

1. List all pods matching the selector that are **Ready**.
2. Write their IPs into an associated `Endpoints` (or `EndpointSlice`) object.
3. `kube-proxy` on every node watches those endpoints and programs the node's networking so the Service's virtual IP forwards to one of the ready pod IPs.

The minimum Service manifest:

```yaml
apiVersion: v1
kind: Service
metadata:
  name: api
spec:
  selector:
    app: api          # picks any pod with this label
  ports:
    - port: 80        # the port the Service exposes
      targetPort: 5678 # the port the container is listening on
```

Notice what's *not* here:

- No list of pod IPs. The Service has zero hardcoded references to any pod.
- No type. It defaults to `ClusterIP` — cluster-internal only.
- No image, no replicas. A Service doesn't run anything; it only routes.

Apply this and the cluster does three things behind your back:

1. Allocates a **ClusterIP** from the service CIDR (typically `10.96.0.0/12`).
2. Registers a **DNS name** — `api.<namespace>.svc.cluster.local` — that resolves to the ClusterIP.
3. Computes the **Endpoints** for the selector and propagates them to every node's `kube-proxy`.

From that moment on, anything in the cluster that knows the name `api` can talk to it.

In [ ]:
# Backing Deployment with three pods labelled app=api, plus a ClusterIP Service in front
!cat <<'EOF' | kubectl apply -f -
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api
spec:
  replicas: 3
  selector:
    matchLabels:
      app: api
  template:
    metadata:
      labels:
        app: api
    spec:
      containers:
        - name: app
          image: hashicorp/http-echo:1.0
          args: ["-text=hello from $(POD_NAME)", "-listen=:5678"]
          env:
            - name: POD_NAME
              valueFrom:
                fieldRef:
                  fieldPath: metadata.name
          ports:
            - containerPort: 5678
              name: http
---
apiVersion: v1
kind: Service
metadata:
  name: api
spec:
  selector:
    app: api
  ports:
    - port: 80
      targetPort: 5678
EOF
!sleep 10
# The Service has a ClusterIP and a port; no pod IPs visible here
!kubectl get svc api
!echo '---'
# Hit the Service from a one-shot client pod — round-robin lands on different api pods
!kubectl run client -i --rm --restart=Never --image=curlimages/curl:8.10.1 --quiet -- \
  sh -c 'for i in 1 2 3 4 5 6; do curl -s http://api:80; done'

## `port`, `targetPort`, `nodePort` — the three numbers

Service manifests have up to three port numbers. New users blur them together; the CKA expects you to be precise.

```yaml
spec:
  ports:
    - port: 80          # what the Service listens on (cluster-internal)
      targetPort: 5678  # what the pod's container is listening on
      nodePort: 31234   # only with type: NodePort — the port opened on every node
      protocol: TCP     # TCP (default) | UDP | SCTP
      name: http        # optional — required when there's more than one port
```

The mental model:

```
in-cluster client           Service                  Pod
        |       :port          |     :targetPort      |
        |--------------------->|--------------------->|
                                                      container listens here

(NodePort type only adds an extra rule:)

external client       node :nodePort       (jumps in via the
        |       :31234         |            same kube-proxy rules)
        |--------------------->|
```

- **`port`** — the port the Service exposes *to in-cluster clients*. Required.
- **`targetPort`** — the port on the pod the traffic is forwarded to. Defaults to `port` if you omit it. Can be a number *or* a named port from the pod spec (e.g. `targetPort: http` matches `ports[*].name: http` on the container). Named ports are the safer choice — the container can change its port and you only edit one file.
- **`nodePort`** — only used when `type: NodePort` (or `LoadBalancer`, which implies a NodePort). Opens this port on *every* node. Range defaults to `30000-32767`. If you omit it, Kubernetes assigns a free one.

A common mistake: setting `targetPort` to the host port, or to `port`, when the container is actually listening on something else. Always read `targetPort` from the container's `containerPort` field.

## Endpoints and EndpointSlices — what the Service actually points at

The Service object holds the *selector*. The *resolved list of pod IPs* lives in a separate object:

- **`Endpoints`** — the original kind. One per Service, with the same name. Lists every ready pod's IP and port.
- **`EndpointSlice`** — the modern kind (default since Kubernetes 1.21). Each slice holds up to 100 endpoints; large Services get multiple slices. Lower overhead at scale.

Both are filled in automatically — you don't write them. The **endpoints controller** watches Services and pods, and updates them whenever:

- a pod matching the selector becomes Ready
- a pod stops being Ready
- a pod is deleted
- the Service's selector changes

When you read `kubectl get endpoints api`, you're reading the live answer to "which pod IPs is this Service pointing at right now?"

If a Service shows `<none>` for endpoints, that's the single most common Services problem in the wild. It means **no pods match the selector** (or no matching pods are Ready). Almost always the cause is:

- The Service selector and pod labels disagree (typo).
- The pods aren't ready (failing readiness probe, still pulling image).
- The pods are in a different namespace from the Service.

`kubectl describe service <name>` shows the selector and the endpoint addresses side by side — the first stop when something can't connect.

### A Service with no selector

You can write a Service with **no selector at all**, and create the `Endpoints` object yourself:

```yaml
apiVersion: v1
kind: Service
metadata:
  name: external-db
spec:
  ports:
    - port: 5432
---
apiVersion: v1
kind: Endpoints
metadata:
  name: external-db    # same name as the Service
subsets:
  - addresses:
      - ip: 10.0.5.20  # an IP outside the cluster
    ports:
      - port: 5432
```

This is the recipe for **pointing a Service name at an external system** (a managed RDS, a legacy VM). Pods inside the cluster can still talk to `external-db`, and the operator can later swap it for an in-cluster Deployment without touching client code.

In [ ]:
# Endpoints object — same name as the Service
!kubectl get endpoints api
!echo '---'
# Or the modern EndpointSlice — finer-grained, scalable
!kubectl get endpointslices -l kubernetes.io/service-name=api
!echo '---'
# Scale the backing Deployment up — endpoints update within seconds
!kubectl scale deployment api --replicas=5
!sleep 8
!kubectl get endpoints api
!echo '---'
# Scale back down
!kubectl scale deployment api --replicas=3
!sleep 6
!kubectl get endpoints api

## Service types — `ClusterIP`, `NodePort`, `LoadBalancer`, `ExternalName`

A Service's `spec.type` controls *how reachable* it is. There are four values.

### `ClusterIP` (default)

The Service gets a virtual IP allocated from the cluster's service CIDR. **Reachable only from inside the cluster** — pods, the control plane, anything `kube-proxy` programmed. This is what every internal microservice should be.

```yaml
spec:
  type: ClusterIP   # the default; can be omitted
```

### `NodePort`

The Service is reachable on **every node's IP**, on a port from the NodePort range (default `30000-32767`). NodePort *includes* ClusterIP — the virtual IP still exists. From outside, you'd hit `http://<any-node-ip>:31234`.

```yaml
spec:
  type: NodePort
  ports:
    - port: 80
      targetPort: 5678
      nodePort: 31234   # optional; auto-assigned if omitted
```

Use it for **development clusters** and **bare-metal clusters without a load balancer**. In a real cloud cluster you almost never expose NodePort directly — a LoadBalancer or an Ingress sits in front.

### `LoadBalancer`

The Service is exposed through a **cloud-provided external load balancer**. Implicitly creates a NodePort too. The cluster's cloud-controller-manager talks to the cloud API (AWS ELB, GCP forwarding rule, Azure LB) and provisions the LB, then writes its public IP back into `status.loadBalancer.ingress`.

```yaml
spec:
  type: LoadBalancer
  ports:
    - port: 80
      targetPort: 5678
```

On `kind` and `minikube` there's no cloud provider — `LoadBalancer` Services stay `Pending` forever. Tools like `metallb` (bare metal) or `cloud-provider-kind` plug a fake LB in for local clusters.

### `ExternalName`

A different beast — no proxying, no selector, no endpoints. Just a **DNS CNAME** from the Service name to an external hostname:

```yaml
spec:
  type: ExternalName
  externalName: db.legacy.example.com
```

Pods that look up `api` get `db.legacy.example.com` back. Useful for **giving an external dependency a Kubernetes-shaped name** so client manifests look the same in every environment.

### Summary table

| Type | Reachable from | Allocates | Use it for |
|---|---|---|---|
| `ClusterIP` | inside cluster | virtual IP | every internal microservice |
| `NodePort` | inside cluster + every node's IP | virtual IP + node port | dev, bare metal, behind an external LB |
| `LoadBalancer` | inside cluster + cloud LB | virtual IP + node port + LB IP | publicly-exposed services in cloud |
| `ExternalName` | DNS only | CNAME | aliasing an external host |

In [ ]:
# A NodePort Service for the same api Deployment
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  name: api-nodeport
spec:
  type: NodePort
  selector:
    app: api
  ports:
    - port: 80
      targetPort: 5678
      nodePort: 31234
EOF
!sleep 3
!kubectl get svc api-nodeport
!echo '---'
# Reach it through its in-cluster name — kube-proxy still routes on the ClusterIP side
!kubectl run client -i --rm --restart=Never --image=curlimages/curl:8.10.1 --quiet -- \
  curl -s http://api-nodeport:80

## How `kube-proxy` actually routes traffic

The Service's ClusterIP is a *fiction*. There's no process listening on `10.96.42.7`. So what makes the packet arrive at a pod?

**`kube-proxy`** does. It runs as a DaemonSet on every node and watches the API server for Services and Endpoints. When it learns about a new Service, it programs the **node's kernel networking** so packets to the Service IP get rewritten to a real pod IP.

Three modes have existed; you'll meet two in the wild.

### `iptables` mode (long-time default)

`kube-proxy` writes `iptables` rules that DNAT packets destined for the ClusterIP to one of the endpoints, picked at random. Pros: simple, in-kernel, no extra process in the data path. Cons: rule count grows linearly with the number of Services × endpoints, which makes rule updates slower in clusters with tens of thousands of Services.

### `IPVS` mode

`kube-proxy` programs the kernel's IPVS (IP Virtual Server) module instead. Same idea, but with hash-based lookup instead of linear iteration, and richer load-balancing algorithms (round-robin, least-connections, source-hash). Faster at large scale. Available on most modern distributions; enable per-cluster.

### `nftables` mode

Newer alternative on Linux kernels with `nftables` support. Replaces `iptables` with the modern packet-filter framework. The direction of travel for new clusters.

### What this means for you

For everyday work, `kube-proxy` is invisible — you write a Service, traffic flows. The cases where the mode matters:

- **Debugging.** `iptables -t nat -L -n` on a node shows the rules `kube-proxy` wrote.
- **Service mesh.** Tools like Istio and Linkerd inject sidecar Envoys that intercept traffic *before* `kube-proxy` sees it. They sit on top of, not instead of, `kube-proxy`.
- **Performance at scale.** Past a few thousand Services, switch from `iptables` to `IPVS` or `nftables`.

### What `kube-proxy` does *not* do

A common misconception: `kube-proxy` is **not** in the data path of an established TCP connection. It only sets up the *rules*. The actual packets flow through the kernel directly between pod and pod, with no userspace hop. That's why Service IPs are essentially free at runtime.

## DNS in Kubernetes — CoreDNS and the service name pattern

A ClusterIP is hard to remember. The piece that makes Services usable is **cluster DNS**. Every Kubernetes cluster ships a DNS server — almost always **CoreDNS** — running as a Deployment in `kube-system`, with its own Service in front.

When a pod starts, the kubelet writes `/etc/resolv.conf` so the pod's DNS resolver points at the CoreDNS Service. Every name lookup the pod makes goes to CoreDNS first.

### The Service DNS pattern

CoreDNS exposes a name for every Service:

```
<service>.<namespace>.svc.cluster.local
```

So a Service called `api` in the `default` namespace resolves at:

```
api.default.svc.cluster.local
```

Three handy shortcuts apply because of the search-domain configuration in `/etc/resolv.conf`:

| From a pod in... | Reaches `api` in `default` as... |
|---|---|
| `default` | `api`, `api.default`, `api.default.svc`, `api.default.svc.cluster.local` |
| `team-a` | `api.default`, `api.default.svc`, ... — must include the namespace |
| `kube-system` | same — namespace prefix required when crossing namespaces |

Two takeaways:

- **Inside the same namespace, the service name alone is enough.** `curl http://api` works.
- **Across namespaces, you must include the namespace.** `curl http://api.default` (or longer).

### What gets a DNS record

- **Services** → `A` / `AAAA` records pointing at the ClusterIP.
- **Headless Services** → `A` / `AAAA` records pointing at every Ready pod's IP. (Section below.)
- **Named ports on a Service** → `SRV` records (`_http._tcp.api.default.svc.cluster.local`).
- **Pods, individually** → `<pod-ip-with-dashes>.<namespace>.pod.cluster.local`, but only opt-in via `spec.hostname` / `spec.subdomain`. Day-to-day, you don't use pod DNS — you use Service DNS.

### Picking a different cluster domain

The `cluster.local` suffix is configurable per-cluster (`kubelet --cluster-domain=...`), but in practice everyone leaves it as `cluster.local`. The CKA assumes `cluster.local`.

### When DNS is the culprit

Symptoms: pods can hit Services by ClusterIP but `nslookup` fails. First checks:

1. `kubectl get pods -n kube-system -l k8s-app=kube-dns` — are CoreDNS pods Running and Ready?
2. `kubectl get svc -n kube-system kube-dns` — is the DNS Service present and does it have endpoints?
3. From a debug pod, `cat /etc/resolv.conf` — is the nameserver pointing at the kube-dns ClusterIP?

We dive into the full troubleshooting flow in notebook 10.

In [ ]:
# Look up the api Service from a one-shot debug pod
!kubectl run dns-test -i --rm --restart=Never --image=busybox:1.36 --quiet -- \
  sh -c 'echo "short name:"; nslookup api; echo; echo "FQDN:"; nslookup api.default.svc.cluster.local'
!echo '---'
# The pod's /etc/resolv.conf — note the cluster.local search domain
!kubectl run dns-test -i --rm --restart=Never --image=busybox:1.36 --quiet -- cat /etc/resolv.conf

## Headless Services — DNS to pod IPs, no virtual IP

Sometimes you don't want a virtual IP. You want clients to see **the individual pod IPs** so they can connect to a specific replica (or all of them). That's a **headless Service**.

You make one by setting `clusterIP: None`:

```yaml
apiVersion: v1
kind: Service
metadata:
  name: api
spec:
  clusterIP: None     # the magic word — "headless"
  selector:
    app: api
  ports:
    - port: 80
      targetPort: 5678
```

Effects:

- **No ClusterIP allocated.** No `kube-proxy` rules. There's no virtual IP to connect to.
- **DNS for the Service name returns multiple A records** — one per Ready pod. A standard DNS-lookup-then-pick-one client (Java's default resolver, Go's `net.LookupHost`) gets every pod's IP.
- **EndpointSlices still exist.** The endpoints controller still tracks ready pods; `kubectl get endpoints` still shows them.

Two use cases dominate.

### StatefulSets

When a StatefulSet wants each pod individually addressable — `pod-0`, `pod-1`, `pod-2` — the canonical setup is a headless Service alongside the StatefulSet. CoreDNS then exposes per-pod DNS:

```
<pod-name>.<service-name>.<namespace>.svc.cluster.local
```

That's how databases like PostgreSQL and Cassandra running in Kubernetes find their peers: `cassandra-0.cassandra.default.svc.cluster.local`, `cassandra-1.cassandra.default.svc.cluster.local`, and so on. Notebook 06 comes back to this.

### Client-side load balancing

Some clients (gRPC, Kafka, anything with a connection pool) want to manage their own connection lifecycle to every backend, not have `kube-proxy` round-robin TCP connections behind their back. They look up the headless Service, learn every endpoint IP, and open a long-lived connection to each.

### A regular Service is the right default

If you don't have a specific reason to need a headless Service, use a regular `ClusterIP`. Headless is the building block under StatefulSets and a tool for specific client patterns — not a general improvement.

In [ ]:
!cat <<'EOF' | kubectl apply -f -
apiVersion: v1
kind: Service
metadata:
  name: api-headless
spec:
  clusterIP: None
  selector:
    app: api
  ports:
    - port: 80
      targetPort: 5678
EOF
!sleep 3
# No ClusterIP — column shows "None"
!kubectl get svc api-headless
!echo '---'
# DNS lookup returns multiple A records, one per Ready pod
!kubectl run dns-test -i --rm --restart=Never --image=busybox:1.36 --quiet -- \
  nslookup api-headless.default.svc.cluster.local

## Session affinity & multi-port services

### Session affinity

By default, every connection to a Service is independently load-balanced — the same client might hit different pods on back-to-back requests. For protocols where that's wrong (long-lived sessions stored in pod-local memory, sticky shopping carts), set:

```yaml
spec:
  sessionAffinity: ClientIP
  sessionAffinityConfig:
    clientIP:
      timeoutSeconds: 10800   # default 3 hours
```

`ClientIP` mode hashes by the client's source IP and pins them to one pod for `timeoutSeconds`. It's the *only* affinity mode Kubernetes offers — for richer session stickiness (cookie-based, JWT-based), you need an Ingress controller or a service mesh.

A warning: when many clients sit behind one NAT (a corporate proxy, mobile carrier CGNAT), `ClientIP` affinity stops being useful — every "client" looks the same. Use it for *internal* traffic where each caller has a distinct IP.

### Multi-port services

A Service can expose more than one port. Each entry in `spec.ports[]` becomes a separate routing rule. When you have more than one port, **each port must have a name**:

```yaml
spec:
  selector:
    app: api
  ports:
    - name: http
      port: 80
      targetPort: 8080
    - name: grpc
      port: 9090
      targetPort: 9090
    - name: metrics
      port: 9091
      targetPort: 9091
```

DNS still gives one set of `A` records (it's still one Service IP); `SRV` records carry the per-port detail. Clients connect to the right port by number, as usual.

## Service vs Ingress — a glance at notebook 09

A Service exposes a workload on one or more L4 ports (TCP, UDP, SCTP). It speaks **the transport layer**: hostnames, paths, TLS, HTTP routing — none of those exist at the Service layer.

For HTTP routing — "send `/api/*` to one Service, `/static/*` to another, with TLS on the front and a single public IP" — you want an **Ingress** (or its newer cousin, the **Gateway API**). An Ingress is itself a Kubernetes object; an **ingress controller** (nginx, Traefik, HAProxy, Istio Gateway, etc.) reads it and configures an HTTP proxy in front of your Services.

A rough mental map:

| Layer | Object | Speaks |
|---|---|---|
| L4 inside the cluster | Service (`ClusterIP`) | TCP/UDP, selector-based |
| L4 from outside | Service (`NodePort` / `LoadBalancer`) | TCP/UDP at the edge |
| L7 HTTP | Ingress + ingress controller | HTTP host/path routing, TLS termination |
| L7 mesh | Service mesh (Istio, Linkerd) | mTLS, retries, traffic-shifting, observability |

Notebook 09 covers Ingress and NetworkPolicy properly. For now, the takeaway: **Service is the foundational primitive.** Everything higher up — Ingress, mesh, the Gateway API — *uses* Services to find the pods, then adds smarter routing on top.

## Cleaning up

Delete everything this notebook created so the cluster's tidy for notebook 05:

```bash
kubectl delete deployment api
kubectl delete service api api-nodeport api-headless
```

Or all in one go:

```bash
kubectl delete deployment,service --all
```